# AI LÀ AI

Notebook baseline tạo trực tiếp `submission.zip` cho tập kiểm tra.

## 1. Cấu hình

In [ ]:
import os

# Phải đặt trước khi torch khởi tạo CUDA.
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

from pathlib import Path
import random
import time
import zipfile
from torchvision import models
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2

DATA_ROOT = Path('data')
SPLIT = 'private_test'        # đổi thành 'private_test' ở giai đoạn kiểm tra bí mật

EPOCHS = 3
BATCH_SIZE = 128
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Cố định kết quả giữa các lần chạy: cùng seed -> cùng file nộp.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[cấu hình] split={SPLIT} | epochs={EPOCHS} | batch={BATCH_SIZE} | ảnh={IMAGE_SIZE}px | seed={SEED}')
print(f'[cấu hình] thiết bị: {DEVICE}'
      + (f' ({torch.cuda.get_device_name(0)})' if DEVICE.type == 'cuda' else ''))

## 2. Mô hình và phép biến đổi ảnh

In [ ]:
backbon = models.densenet121(weights='DEFAULT')

In [ ]:
class CNN2(nn.Module):
    """CNN hai khối tích chập, phân loại ảnh thật/giả."""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(32, 2),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


normalize = transforms.Normalize((.485, .456, .406), (.229, .224, .225))

train_tf = v2.Compose([
    v2.ToImage(),
    v2.Resize(224, 224),
    v2.RandomResizedCrop((224, 224)),
    v2.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5.)),
    v2.RandomHorizontalFlip(),
    v2.RandomPerspective(distortion_scale=0.2, p=0.3),
    v2.RandomAffine(15, translate=(0.1, 0.1), shear=(8)),
    v2.ColorJitter(0.2, 0.2, 0.2),
    v2.ToDtype(torch.float, scale=True),
    v2.Normalize((.485, .456, .406), (.229, .224, .225))
])
test_tf = v2.Compose([
    v2.ToImage(),
    v2.Resize((224, 224)),
    v2.ToDtype(torch.float, scale=True),
    v2.Normalize((.485, .456, .406), (.229, .224, .225))
])

print(f'[mô hình] CNN2: {sum(p.numel() for p in CNN2().parameters()):,} tham số')

## 3. Dataset

In [ ]:
def load_image(path: Path, tf):
    with Image.open(path) as image:
        return tf(ImageOps.exif_transpose(image).convert('RGB'))


class FaceImages(Dataset):
    """Đọc ảnh khuôn mặt; có nhãn khi huấn luyện, trả tên file khi dự đoán."""

    def __init__(self, rows, root, tf, labeled=True):
        self.rows = rows.reset_index(drop=True)
        self.root = root
        self.tf = tf
        self.labeled = labeled

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        name = Path(row.path).name if self.labeled else row.file_name
        image = load_image(self.root / 'images' / name, self.tf)
        return (image, int(row.label)) if self.labeled else (image, name)


print('[dataset] đã định nghĩa FaceImages (dùng chung cho train và dự đoán)')

## 4. Nạp dữ liệu

In [ ]:
df = pd.read_csv(DATA_ROOT / 'train' / 'manifest.csv')
train_df = df.sample(int(0.8 * len(df)), random_state=42)
val_df = df.drop(index=train_df.index)

train_data = FaceImages(train_df, DATA_ROOT / 'train', train_tf)
val_data = FaceImages(val_df, DATA_ROOT / 'train', test_tf)
query = pd.DataFrame({
    'file_name': sorted(p.name for p in (DATA_ROOT / SPLIT / 'images').iterdir() if p.is_file())
})

train_loader = DataLoader(
    train_data , batch_size=BATCH_SIZE, shuffle=True
)

val_loader = DataLoader(
    val_data , batch_size=BATCH_SIZE, shuffle=False
)
print(f'[dữ liệu] train: {len(train_data):,} ảnh')
print(f'[dữ liệu] val: {len(val_data):,} ảnh')
print(f'[dữ liệu] {SPLIT}: {len(query):,} ảnh cần dự đoán')
print(f'[dữ liệu] phân bố nhãn train: {train_data.label.value_counts().to_dict()}')

In [ ]:
import matplotlib.pyplot as plt
imgs, labels = next(iter(train_loader))
fig, ax = plt.subplots(3, 4)
mean = torch.tensor([0.485, 0.456, 0.406]).reshape(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).reshape(3, 1, 1)

for i in range(12):
    img, label = imgs[i], labels[i]
    img = img * std + mean
    img = img.permute(1,2,0).numpy()
    ax[i].imshow(img)
    ax[i].set_title(f"label {label}", fontsize=8)
    ax[i].axis('off')
plt.show()

## 5. Huấn luyện

In [ ]:
model = CNN2().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(1, EPOCHS + 1):
    model.train()
    started = time.time()
    total_loss = 0.0
    correct = 0
    seen = 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        seen += labels.size(0)

    print(f'[huấn luyện] epoch {epoch}/{EPOCHS} | loss={total_loss / seen:.4f} '
          f'| accuracy={correct / seen:.4f} | {time.time() - started:.1f}s')

print('[huấn luyện] hoàn tất')

## 6. Dự đoán và đóng gói bài nộp

In [ ]:
model.eval()
rows = []
started = time.time()

with torch.inference_mode():
    loader = DataLoader(
        FaceImages(query, DATA_ROOT / SPLIT, eval_tf, labeled=False), batch_size=BATCH_SIZE
    )
    for images, names in loader:
        labels = model(images.to(DEVICE)).argmax(1).cpu().tolist()
        rows.extend(zip(names, labels))

submission = pd.DataFrame(rows, columns=['file_name', 'category_id'])
submission.to_csv('submission.csv', index=False)

with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write('submission.csv')

print(f'[dự đoán] {len(submission):,} ảnh trong {time.time() - started:.1f}s')
print(f'[dự đoán] phân bố nhãn: {submission.category_id.value_counts().to_dict()}')
print('[nộp bài] đã tạo submission.zip (chứa đúng submission.csv ở thư mục gốc)')